In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Entrenamiento y Validación UPLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial KNN (Naranja)
COLOR_KNN = '#ff7f0e'

configurar_estilo_tesis()

# ==============================================================================
# 1. CARGA DE DATOS (UPLINK)
# ==============================================================================
DIR_DATA = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/UpLink/'
DIR_OUTPUT = '/content/drive/MyDrive/TESIS 100%/KNN/Uplink/Entrenamiento/'

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

print("⚙️ INICIANDO FASE 1: ENTRENAMIENTO KNN UPLINK (ESTILO TESIS)...")

def estandarizar_columnas(df):
    cols_map = {}
    if 'SpreedFactor_UpLink' in df.columns: cols_map['SpreedFactor_UpLink'] = 'SpreedFactor'
    if 'Path_Loss_UpLink' in df.columns: cols_map['Path_Loss_UpLink'] = 'Path_Loss_Uplink'
    if cols_map: df.rename(columns=cols_map, inplace=True)
    return df

try:
    train_df = pd.read_csv(os.path.join(DIR_DATA, 'Train_UpLink.csv'))
    val_df   = pd.read_csv(os.path.join(DIR_DATA, 'Val_UpLink.csv'))
    train_df = estandarizar_columnas(train_df)
    val_df   = estandarizar_columnas(val_df)
    print(f"✅ Datos cargados. Train: {len(train_df)} | Val: {len(val_df)}")
except FileNotFoundError:
    print("❌ Error: Archivos no encontrados."); exit()

FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', 'SpreedFactor', 'Terreno', 'Power_UpLink']
TARGET = 'Path_Loss_Uplink'

# Verificación
for col in FEATURES + [TARGET]:
    if col not in train_df.columns:
        print(f"❌ Falta columna: {col}"); exit()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]

# ==============================================================================
# 2. BUCLE DE ENTRENAMIENTO (BUSCANDO K)
# ==============================================================================
MAX_K = 100
PATIENCE = 15
patience_counter = 0
best_val_rmse = float('inf')
best_k = 1

history = {'k': [], 'train_rmse': [], 'val_rmse': [], 'train_mse': [],
           'val_mse': [], 'train_mae': [], 'val_mae': [], 'train_r2': [], 'val_r2': []}

ruta_modelo_local = '/content/Modelo_KNN_Uplink.pkl'
ruta_modelo_drive = os.path.join(DIR_OUTPUT, 'Modelo_KNN_Uplink.pkl')

print("\n🔄 Optimizando K (Vecinos)...")

for k in range(1, MAX_K + 1):
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance', n_jobs=-1)
    knn.fit(X_train, y_train)

    p_t = knn.predict(X_train)
    p_v = knn.predict(X_val)

    # Métricas
    mse_t, mse_v = mean_squared_error(y_train, p_t), mean_squared_error(y_val, p_v)
    rmse_t, rmse_v = np.sqrt(mse_t), np.sqrt(mse_v)
    mae_t, mae_v = mean_absolute_error(y_train, p_t), mean_absolute_error(y_val, p_v)
    r2_t, r2_v = r2_score(y_train, p_t), r2_score(y_val, p_v)

    history['k'].append(k)
    history['train_rmse'].append(rmse_t); history['val_rmse'].append(rmse_v)
    history['train_mse'].append(mse_t);   history['val_mse'].append(mse_v)
    history['train_mae'].append(mae_t);   history['val_mae'].append(mae_v)
    history['train_r2'].append(r2_t);     history['val_r2'].append(r2_v)

    if k % 10 == 0: print(f"   Neighbors {k} | RMSE Val: {rmse_v:.4f}")

    if rmse_v < best_val_rmse:
        best_val_rmse = rmse_v
        best_k = k
        patience_counter = 0
        joblib.dump(knn, ruta_modelo_local)
        shutil.copy(ruta_modelo_local, ruta_modelo_drive)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n🛑 STOP temprano en K={k}. Mejor K={best_k}"); break

# ==============================================================================
# 3. DASHBOARD 2x2 DE MÉTRICAS (KNN)
# ==============================================================================
print("\n📊 Generando Dashboard de Métricas...")
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'KNN Uplink: Curvas de Aprendizaje (Óptimo: K={best_k})', fontweight='bold')

metrics_config = [
    ('RMSE', 'train_rmse', 'val_rmse', axs[0, 0], 'Error (dB)'),
    ('MSE', 'train_mse', 'val_mse', axs[0, 1], 'Error Cuadrático (dB²)'),
    ('MAE', 'train_mae', 'val_mae', axs[1, 0], 'Error Absoluto (dB)'),
    ('R²', 'train_r2', 'val_r2', axs[1, 1], 'Score (0-1)')
]

idx_opt = history['k'].index(best_k)

for name, key_t, key_v, ax, y_label in metrics_config:
    ax.plot(history['k'], history[key_t], '--', color='gray', label='Train', alpha=0.7)
    ax.plot(history['k'], history[key_v], color=COLOR_KNN, label='Validación', linewidth=2.5)

    val_opt = history[key_v][idx_opt]
    ax.axvline(x=best_k, color='black', linestyle=':', alpha=0.5)
    ax.scatter([best_k], [val_opt], color='black', zorder=5)

    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Número de Vecinos (k)', fontweight='bold') # Eje X específico KNN
    ax.set_ylabel(y_label, fontweight='bold')
    ax.legend(loc='best')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(DIR_OUTPUT, '1_Dashboard_Metricas.png'), dpi=300)
plt.close()

# ==============================================================================
# 4. ANÁLISIS DE ERROR AVANZADO
# ==============================================================================
print("🔍 Generando Gráficas de Análisis...")
best_knn = joblib.load(ruta_modelo_drive)
p_val = best_knn.predict(X_val)
residuos = y_val - p_val

# --- A. Dispersión Real vs Predicho ---
plt.figure(figsize=(8, 8))
plt.scatter(y_val, p_val, alpha=0.5, color=COLOR_KNN, edgecolor='w', s=50)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--', lw=2, color='black', label='Ideal')
plt.title(f'KNN (K={best_k}): Real vs Predicho', fontweight='bold')
plt.xlabel('Path Loss Real (dB)', fontweight='bold')
plt.ylabel('Path Loss Predicho (dB)', fontweight='bold')
plt.legend()
plt.savefig(os.path.join(DIR_OUTPUT, '2_Dispersion_Real_vs_Predicho.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- B. Histograma de Errores ---
plt.figure(figsize=(10, 6))
sns.histplot(residuos, kde=True, color=COLOR_KNN, bins=30, line_kws={'linewidth': 2})
plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
plt.title(f'Distribución de Errores (Media: {residuos.mean():.2f} dB)', fontweight='bold')
plt.xlabel('Error (dB)', fontweight='bold'); plt.ylabel('Frecuencia', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '3_Distribucion_Errores.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- C. Homocedasticidad ---
plt.figure(figsize=(10, 6))
plt.scatter(p_val, residuos, alpha=0.5, color=COLOR_KNN, edgecolor='w')
plt.axhline(y=0, color='black', linestyle='--', lw=2)
plt.title('Homocedasticidad: Residuos vs Predicción', fontweight='bold')
plt.xlabel('Path Loss Predicho (dB)', fontweight='bold'); plt.ylabel('Residuo (dB)', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '4_Homocedasticidad.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- D. QQ-Plot ---
plt.figure(figsize=(10, 6))
stats.probplot(residuos, dist="norm", plot=plt)
plt.title('QQ-Plot: Normalidad de Residuos', fontweight='bold')
plt.xlabel('Cuantiles Teóricos', fontweight='bold'); plt.ylabel('Valores Ordenados', fontweight='bold')
plt.gca().get_lines()[0].set_markerfacecolor(COLOR_KNN)
plt.gca().get_lines()[0].set_markeredgecolor(COLOR_KNN)
plt.gca().get_lines()[1].set_color('black')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig(os.path.join(DIR_OUTPUT, '5_QQ_Plot.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- E. CDF del Error Absoluto ---
sorted_abs_error = np.sort(np.abs(residuos))
p = 1. * np.arange(len(sorted_abs_error)) / (len(sorted_abs_error) - 1)
plt.figure(figsize=(10, 6))
plt.plot(sorted_abs_error, p, linewidth=3, color=COLOR_KNN)
plt.title('CDF: Probabilidad Acumulada del Error Absoluto', fontweight='bold')
plt.xlabel('Error Absoluto (dB)', fontweight='bold'); plt.ylabel('Probabilidad', fontweight='bold')
idx_90 = np.searchsorted(p, 0.90)
err_90 = sorted_abs_error[idx_90]
plt.axvline(x=err_90, color='black', linestyle=':')
plt.text(err_90, 0.5, f' 90% < {err_90:.2f} dB', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '6_CDF_Error.png'), dpi=300, bbox_inches='tight'); plt.close()

# ==============================================================================
# 5. TABLA DE RESULTADOS FINAL
# ==============================================================================
print("📋 Generando Tabla de Resultados...")

idx = history['k'].index(best_k)
best_rmse = history['val_rmse'][idx]
best_mse = history['val_mse'][idx]
best_mae = history['val_mae'][idx]
best_r2 = history['val_r2'][idx]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('tight'); ax.axis('off')

data_table = [
    ['Métrica (KNN Uplink)', 'Valor Óptimo'],
    ['K (Vecinos)', f"{best_k}"],
    ['RMSE', f"{best_rmse:.4f} dB"],
    ['MAE', f"{best_mae:.4f} dB"],
    ['MSE', f"{best_mse:.4f} dB²"],
    ['R² Score', f"{best_r2:.4f}"]
]

table = ax.table(cellText=data_table, colLabels=None, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 1.8)

# Estilo Naranja
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(COLOR_KNN) # Cabecera Naranja
    else:
        cell.set_facecolor('#f8f9fa')
        cell.set_edgecolor('#dddddd')

plt.title('Resumen de Desempeño KNN (Uplink)', fontsize=14, fontweight='bold', y=0.95)
plt.savefig(os.path.join(DIR_OUTPUT, '7_Tabla_Metricas_Finales.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Procesamiento terminado. Gráficas y Tabla guardadas en: {DIR_OUTPUT}")

# **Entrenamiento y Validación DOWNLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial KNN (Naranja)
COLOR_KNN = '#ff7f0e'

configurar_estilo_tesis()

# ==============================================================================
# 1. CARGA DE DATOS (DOWNLINK)
# ==============================================================================
DIR_DATA = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/DownLink/'
DIR_OUTPUT = '/content/drive/MyDrive/TESIS 100%/KNN/Downlink/Entrenamiento/'

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

print("⚙️ INICIANDO FASE 1: ENTRENAMIENTO KNN DOWNLINK (ESTILO TESIS)...")

def estandarizar_columnas_dl(df):
    # Lógica robusta para encontrar SF y PathLoss
    cols_map = {}

    # Buscamos SF
    posibles_sf = ['SpreedFactor_UpLink', 'SpreedFactor_DownLink', 'SpreedFactor', 'SF']
    sf_found = next((c for c in posibles_sf if c in df.columns), None)
    if sf_found and sf_found != 'SpreedFactor':
        cols_map[sf_found] = 'SpreedFactor'

    # Buscamos PathLoss
    posibles_pl = ['Path_Loss_DownLink', 'Path_Loss_Downlink', 'Path_Loss']
    pl_found = next((c for c in posibles_pl if c in df.columns), None)
    if pl_found and pl_found != 'Path_Loss_DownLink':
        cols_map[pl_found] = 'Path_Loss_DownLink'

    if cols_map:
        df.rename(columns=cols_map, inplace=True)
        print(f"   ℹ️ Columnas renombradas: {cols_map}")
    return df

try:
    train_df = pd.read_csv(os.path.join(DIR_DATA, 'Train_DownLink.csv'))
    val_df   = pd.read_csv(os.path.join(DIR_DATA, 'Val_DownLink.csv'))
    train_df = estandarizar_columnas_dl(train_df)
    val_df   = estandarizar_columnas_dl(val_df)
    print(f"✅ Datos cargados. Train: {len(train_df)} | Val: {len(val_df)}")
except FileNotFoundError:
    print("❌ Error: Archivos no encontrados."); exit()

FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', 'SpreedFactor', 'Terreno', 'Power_DownLink']
TARGET = 'Path_Loss_DownLink'

# Verificación
for col in FEATURES + [TARGET]:
    if col not in train_df.columns:
        print(f"❌ Falta columna: {col}"); exit()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val     = val_df[FEATURES], val_df[TARGET]

# ==============================================================================
# 2. BUCLE DE ENTRENAMIENTO (BUSCANDO K)
# ==============================================================================
MAX_K = 100
PATIENCE = 15
patience_counter = 0
best_val_rmse = float('inf')
best_k = 1

history = {'k': [], 'train_rmse': [], 'val_rmse': [], 'train_mse': [],
           'val_mse': [], 'train_mae': [], 'val_mae': [], 'train_r2': [], 'val_r2': []}

ruta_modelo_local = '/content/Modelo_KNN_Downlink.pkl'
ruta_modelo_drive = os.path.join(DIR_OUTPUT, 'Modelo_KNN_Downlink.pkl')

print("\n🔄 Optimizando K (Vecinos)...")

for k in range(1, MAX_K + 1):
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance', n_jobs=-1)
    knn.fit(X_train, y_train)

    p_t = knn.predict(X_train)
    p_v = knn.predict(X_val)

    # Métricas
    mse_t, mse_v = mean_squared_error(y_train, p_t), mean_squared_error(y_val, p_v)
    rmse_t, rmse_v = np.sqrt(mse_t), np.sqrt(mse_v)
    mae_t, mae_v = mean_absolute_error(y_train, p_t), mean_absolute_error(y_val, p_v)
    r2_t, r2_v = r2_score(y_train, p_t), r2_score(y_val, p_v)

    history['k'].append(k)
    history['train_rmse'].append(rmse_t); history['val_rmse'].append(rmse_v)
    history['train_mse'].append(mse_t);   history['val_mse'].append(mse_v)
    history['train_mae'].append(mae_t);   history['val_mae'].append(mae_v)
    history['train_r2'].append(r2_t);     history['val_r2'].append(r2_v)

    if k % 10 == 0: print(f"   Neighbors {k} | RMSE Val: {rmse_v:.4f}")

    if rmse_v < best_val_rmse:
        best_val_rmse = rmse_v
        best_k = k
        patience_counter = 0
        joblib.dump(knn, ruta_modelo_local)
        shutil.copy(ruta_modelo_local, ruta_modelo_drive)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n🛑 STOP temprano en K={k}. Mejor K={best_k}"); break

# ==============================================================================
# 3. DASHBOARD 2x2 DE MÉTRICAS (KNN DL)
# ==============================================================================
print("\n📊 Generando Dashboard de Métricas...")
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'KNN Downlink: Curvas de Aprendizaje (Óptimo: K={best_k})', fontweight='bold')

metrics_config = [
    ('RMSE', 'train_rmse', 'val_rmse', axs[0, 0], 'Error (dB)'),
    ('MSE', 'train_mse', 'val_mse', axs[0, 1], 'Error Cuadrático (dB²)'),
    ('MAE', 'train_mae', 'val_mae', axs[1, 0], 'Error Absoluto (dB)'),
    ('R²', 'train_r2', 'val_r2', axs[1, 1], 'Score (0-1)')
]

idx_opt = history['k'].index(best_k)

for name, key_t, key_v, ax, y_label in metrics_config:
    ax.plot(history['k'], history[key_t], '--', color='gray', label='Train', alpha=0.7)
    ax.plot(history['k'], history[key_v], color=COLOR_KNN, label='Validación', linewidth=2.5)

    val_opt = history[key_v][idx_opt]
    ax.axvline(x=best_k, color='black', linestyle=':', alpha=0.5)
    ax.scatter([best_k], [val_opt], color='black', zorder=5)

    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Número de Vecinos (k)', fontweight='bold')
    ax.set_ylabel(y_label, fontweight='bold')
    ax.legend(loc='best')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(DIR_OUTPUT, '1_Dashboard_Metricas_DL.png'), dpi=300)
plt.close()

# ==============================================================================
# 4. ANÁLISIS DE ERROR AVANZADO (DL)
# ==============================================================================
print("🔍 Generando Gráficas de Análisis...")
best_knn = joblib.load(ruta_modelo_drive)
p_val = best_knn.predict(X_val)
residuos = y_val - p_val

# --- A. Dispersión Real vs Predicho ---
plt.figure(figsize=(8, 8))
plt.scatter(y_val, p_val, alpha=0.5, color=COLOR_KNN, edgecolor='w', s=50)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--', lw=2, color='black', label='Ideal')
plt.title(f'KNN DL (K={best_k}): Real vs Predicho', fontweight='bold')
plt.xlabel('Path Loss Real (dB)', fontweight='bold')
plt.ylabel('Path Loss Predicho (dB)', fontweight='bold')
plt.legend()
plt.savefig(os.path.join(DIR_OUTPUT, '2_Dispersion_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- B. Histograma de Errores ---
plt.figure(figsize=(10, 6))
sns.histplot(residuos, kde=True, color=COLOR_KNN, bins=30, line_kws={'linewidth': 2})
plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5)
plt.title(f'KNN DL: Distribución de Errores (Media: {residuos.mean():.2f} dB)', fontweight='bold')
plt.xlabel('Error (dB)', fontweight='bold'); plt.ylabel('Frecuencia', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '3_Distribucion_Errores_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- C. Homocedasticidad ---
plt.figure(figsize=(10, 6))
plt.scatter(p_val, residuos, alpha=0.5, color=COLOR_KNN, edgecolor='w')
plt.axhline(y=0, color='black', linestyle='--', lw=2)
plt.title('KNN DL: Homocedasticidad', fontweight='bold')
plt.xlabel('Path Loss Predicho (dB)', fontweight='bold'); plt.ylabel('Residuo (dB)', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '4_Homocedasticidad_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- D. QQ-Plot ---
plt.figure(figsize=(10, 6))
stats.probplot(residuos, dist="norm", plot=plt)
plt.title('KNN DL: QQ-Plot Normalidad', fontweight='bold')
plt.xlabel('Cuantiles Teóricos', fontweight='bold'); plt.ylabel('Valores Ordenados', fontweight='bold')
plt.gca().get_lines()[0].set_markerfacecolor(COLOR_KNN)
plt.gca().get_lines()[0].set_markeredgecolor(COLOR_KNN)
plt.gca().get_lines()[1].set_color('black')
plt.grid(True, linestyle='--', alpha=0.3)
plt.savefig(os.path.join(DIR_OUTPUT, '5_QQ_Plot_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# --- E. CDF del Error Absoluto ---
sorted_abs_error = np.sort(np.abs(residuos))
p = 1. * np.arange(len(sorted_abs_error)) / (len(sorted_abs_error) - 1)
plt.figure(figsize=(10, 6))
plt.plot(sorted_abs_error, p, linewidth=3, color=COLOR_KNN)
plt.title('KNN DL: CDF Error Absoluto', fontweight='bold')
plt.xlabel('Error Absoluto (dB)', fontweight='bold'); plt.ylabel('Probabilidad', fontweight='bold')
idx_90 = np.searchsorted(p, 0.90)
err_90 = sorted_abs_error[idx_90]
plt.axvline(x=err_90, color='black', linestyle=':')
plt.text(err_90, 0.5, f' 90% < {err_90:.2f} dB', fontweight='bold')
plt.savefig(os.path.join(DIR_OUTPUT, '6_CDF_Error_DL.png'), dpi=300, bbox_inches='tight'); plt.close()

# ==============================================================================
# 5. TABLA DE RESULTADOS FINAL (DL)
# ==============================================================================
print("📋 Generando Tabla de Resultados...")

idx = history['k'].index(best_k)
best_rmse = history['val_rmse'][idx]
best_mse = history['val_mse'][idx]
best_mae = history['val_mae'][idx]
best_r2 = history['val_r2'][idx]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('tight'); ax.axis('off')

data_table = [
    ['Métrica (KNN Downlink)', 'Valor Óptimo'],
    ['K (Vecinos)', f"{best_k}"],
    ['RMSE', f"{best_rmse:.4f} dB"],
    ['MAE', f"{best_mae:.4f} dB"],
    ['MSE', f"{best_mse:.4f} dB²"],
    ['R² Score', f"{best_r2:.4f}"]
]

table = ax.table(cellText=data_table, colLabels=None, cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(12); table.scale(1, 1.8)

# Estilo Naranja
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor(COLOR_KNN) # Cabecera Naranja
    else:
        cell.set_facecolor('#f8f9fa')
        cell.set_edgecolor('#dddddd')

plt.title('Resumen de Desempeño KNN (Downlink)', fontsize=14, fontweight='bold', y=0.95)
plt.savefig(os.path.join(DIR_OUTPUT, '7_Tabla_Metricas_Finales_DL.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Procesamiento KNN Downlink terminado. Todo guardado en: {DIR_OUTPUT}")